# 层和块
:label:`sec_model_construction`

之前首次介绍神经网络时，我们关注的是具有单一输出的线性模型。
在这里，整个模型只有一个输出。
注意，单个神经网络
（1）接受一些输入；
（2）生成相应的标量输出；
（3）具有一组相关 *参数*（parameters），更新这些参数可以优化某目标函数。

然后，当考虑具有多个输出的网络时，
我们利用矢量化算法来描述整层神经元。
像单个神经元一样，层（1）接受一组输入，
（2）生成相应的输出，
（3）由一组可调整参数描述。
当我们使用softmax回归时，一个单层本身就是模型。
然而，即使我们随后引入了多层感知机，我们仍然可以认为该模型保留了上面所说的基本架构。

对于多层感知机而言，整个模型及其组成层都是这种架构。
整个模型接受原始输入（特征），生成输出（预测），
并包含一些参数（所有组成层的参数集合）。
同样，每个单独的层接收输入（由前一层提供），
生成输出（到下一层的输入），并且具有一组可调参数，
这些参数根据从下一层反向传播的信号进行更新。

事实证明，研究讨论“比单个层大”但“比整个模型小”的组件更有价值。
例如，在计算机视觉中广泛流行的ResNet-152架构就有数百层，
这些层是由*层组*（groups of layers）的重复模式组成。
这个ResNet架构赢得了2015年ImageNet和COCO计算机视觉比赛
的识别和检测任务 :cite:`He.Zhang.Ren.ea.2016`。
目前ResNet架构仍然是许多视觉任务的首选架构。
在其他的领域，如自然语言处理和语音，
层组以各种重复模式排列的类似架构现在也是普遍存在。

为了实现这些复杂的网络，我们引入了神经网络*块*的概念。
*块*（block）可以描述单个层、由多个层组成的组件或整个模型本身。
使用块进行抽象的一个好处是可以将一些块组合成更大的组件，
这一过程通常是递归的，如 :numref:`fig_blocks`所示。
通过定义代码来按需生成任意复杂度的块，
我们可以通过简洁的代码实现复杂的神经网络。

![多个层被组合成块，形成更大的模型](../img/blocks.svg)
:label:`fig_blocks`

从编程的角度来看，块由*类*（class）表示。
它的任何子类都必须定义一个将其输入转换为输出的前向传播函数，
并且必须存储任何必需的参数。
注意，有些块不需要任何参数。
最后，为了计算梯度，块必须具有反向传播函数。
在定义我们自己的块时，由于自动微分（在 :numref:`sec_autograd` 中引入）
提供了一些后端实现，我们只需要考虑前向传播函数和必需的参数。

在构造自定义块之前，(**我们先回顾一下多层感知机**)
（ :numref:`sec_mlp_concise` ）的代码。
下面的代码生成一个网络，其中包含一个具有256个单元和ReLU激活函数的全连接隐藏层，
然后是一个具有10个隐藏单元且不带激活函数的全连接输出层。


In [15]:
import torch
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))

X = torch.rand(2, 20)
net(X)

tensor([[ 0.2520,  0.2805, -0.1218, -0.2536, -0.1746, -0.1393, -0.1889, -0.0266,
          0.0364,  0.1518],
        [ 0.0872,  0.1331, -0.1215, -0.2860, -0.1076, -0.0482,  0.0049,  0.0136,
          0.0527,  0.1633]], grad_fn=<AddmmBackward0>)

In [16]:
#grad_fn=<AddmmBackward0>
# 表示这个结果保留了计算过程，PyTorch 可以在之后执行反向传播，计算模型参数的梯度。

在这个例子中，我们通过实例化`nn.Sequential`来构建我们的模型，
层的执行顺序是作为参数传递的。
简而言之，(**`nn.Sequential`定义了一种特殊的`Module`**)，
即在PyTorch中表示一个块的类，
它维护了一个由`Module`组成的有序列表。
注意，两个全连接层都是`Linear`类的实例，
`Linear`类本身就是`Module`的子类。
另外，到目前为止，我们一直在通过`net(X)`调用我们的模型来获得模型的输出。
这实际上是`net.__call__(X)`的简写。
这个前向传播函数非常简单：
它将列表中的每个块连接在一起，将每个块的输出作为下一个块的输入。


## [**自定义块**]

要想直观地了解块是如何工作的，最简单的方法就是自己实现一个。
在实现我们自定义块之前，我们简要总结一下每个块必须提供的基本功能。


1. 将输入数据作为其前向传播函数的参数。
1. 通过前向传播函数来生成输出。请注意，输出的形状可能与输入的形状不同。例如，我们上面模型中的第一个全连接的层接收一个20维的输入，但是返回一个维度为256的输出。
1. 计算其输出关于输入的梯度，可通过其反向传播函数进行访问。通常这是自动发生的。
1. 存储和访问前向传播计算所需的参数。
1. 根据需要初始化模型参数。


在下面的代码片段中，我们从零开始编写一个块。
它包含一个多层感知机，其具有256个隐藏单元的隐藏层和一个10维输出层。
注意，下面的`MLP`类继承了表示块的类。
我们的实现只需要提供我们自己的构造函数（Python中的`__init__`函数）和前向传播函数。


In [17]:
class MLP(nn.Module):       #定义一个名为 MLP 的类，并继承 nn.Module。
                            #nn.Module 是 PyTorch 所有神经网络层和模型的基类
    # 用模型参数声明层。这里，我们声明两个全连接的层
    def __init__(self):
        # 调用MLP的父类Module的构造函数来执行必要的初始化。
        # 这样，在类实例化时也可以指定其他函数参数，例如模型参数params（稍后将介绍）
        super().__init__()     #调用父类 nn.Module 的构造函数
        self.hidden = nn.Linear(20, 256)  # 隐藏层
        self.out = nn.Linear(256, 10)  # 输出层

    # 定义模型的前向传播，即如何根据输入X返回所需的模型输出
    def forward(self, X):
        # 注意，这里我们使用ReLU的函数版本，其在nn.functional模块中定义。
        return self.out(F.relu(self.hidden(X)))

In [57]:
#调用这个类的例子
net = MLP()              # 创建模型，调用 __init__()

K = torch.rand(2, 20)    # 2个样本，每个样本20个特征
M = net(K)               # 调用模型

print(M.shape)            # torch.Size([2, 10])
#查看模型参数、结构
print(net)

for name, param in net.named_parameters():
    print(name, param.shape)

torch.Size([2, 10])
MLP(
  (hidden): Linear(in_features=20, out_features=256, bias=True)
  (out): Linear(in_features=256, out_features=10, bias=True)
)
hidden.weight torch.Size([256, 20])
hidden.bias torch.Size([256])
out.weight torch.Size([10, 256])
out.bias torch.Size([10])


我们首先看一下前向传播函数，它以`X`作为输入，
计算带有激活函数的隐藏表示，并输出其未规范化的输出值。
在这个`MLP`实现中，两个层都是实例变量。
要了解这为什么是合理的，可以想象实例化两个多层感知机（`net1`和`net2`），
并根据不同的数据对它们进行训练。
当然，我们希望它们学到两种不同的模型。

接着我们[**实例化多层感知机的层，然后在每次调用前向传播函数时调用这些层**]。
注意一些关键细节：
首先，我们定制的`__init__`函数通过`super().__init__()`
调用父类的`__init__`函数，
省去了重复编写模版代码的痛苦。
然后，我们实例化两个全连接层，
分别为`self.hidden`和`self.out`。
注意，除非我们实现一个新的运算符，
否则我们不必担心反向传播函数或参数初始化，
系统将自动生成这些。

我们来试一下这个函数：


In [27]:
net = MLP()
net(X)

tensor([[ 0.1787,  0.1011,  0.0406,  0.1897,  0.2159, -0.0445,  0.0562, -0.0409,
          0.1678, -0.0870],
        [ 0.1679,  0.0897,  0.1176,  0.0054,  0.1750,  0.0232,  0.0416, -0.0626,
          0.2424,  0.0458]], grad_fn=<AddmmBackward0>)

块的一个主要优点是它的多功能性。
我们可以子类化块以创建层（如全连接层的类）、
整个模型（如上面的`MLP`类）或具有中等复杂度的各种组件。
我们在接下来的章节中充分利用了这种多功能性，
比如在处理卷积神经网络时。

## [**顺序块**]

现在我们可以更仔细地看看`Sequential`类是如何工作的，
回想一下`Sequential`的设计是为了把其他模块串起来。
为了构建我们自己的简化的`MySequential`，
我们只需要定义两个关键函数：

1. 一种将块逐个追加到列表中的函数；
1. 一种前向传播函数，用于将输入按追加块的顺序传递给块组成的“链条”。

下面的`MySequential`类提供了与默认`Sequential`类相同的功能。


In [42]:
class MySequential(nn.Module):
    def __init__(self, *args):           #*args表示可变参数，可以接受任意数量的参数
        super().__init__()
        for idx, module in enumerate(args):   ##enumerate()函数返回一个包含键值对的#元组#序列，
                                                ##其中键是索引，值是对应元素（见下一个cell）
            # idx是索引，module是对应元素
            # 我们把module保存在'Module'类的成员变量_modules中。_modules的类型是OrderedDict
            # 这里，module是Module子类的一个实例。我们把它保存在'Module'类的成员
            # 变量_modules中。_module的类型是OrderedDict
            self._modules[str(idx)] = module  #这一步注册子模块（见第二个注释cell）

            #这段代码是为了讲解原理。实际开发中不建议直接操作 _modules 这个内部属性，更常见的是
            #self.add_module(str(idx), module)

    def forward(self, X):     ##定义输入如何经过网络。
        # OrderedDict保证了按照成员添加的顺序遍历它们
        for block in self._modules.values():
            X = block(X)
        return X

# 实际开发中
# 需要列表形式使用nn.ModuleList

In [43]:
#enumerate(args) 会同时给出元素的索引和元素本身。
# 举一个arg的例子
# args = (
#    nn.Linear(20, 256),
#    nn.ReLU(),
#    nn.Linear(256, 10)
#)
#解释这个循环
# for idx, module in enumerate(args):
#idx = 0，module = nn.Linear(20, 256)
#idx = 1，module = nn.ReLU()
#idx = 2，module = nn.Linear(256, 10)

In [44]:
#self._modules = {
#    "0": nn.Linear(20, 256),
#    "1": nn.ReLU(),
#    "2": nn.Linear(256, 10)
#}
##modules是Module类的成员变量，类型是OrderedDict
net_1 = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net_1._modules


OrderedDict([('0', Linear(in_features=20, out_features=256, bias=True)),
             ('1', ReLU()),
             ('2', Linear(in_features=256, out_features=10, bias=True))])

In [45]:
#如果确实需要以列表形式保存模块，不能简单写成：
#self.layers = [
#    nn.Linear(20, 256),
#    nn.ReLU(),
#    nn.Linear(256, 10)
#]
#因为这样做不会将这些模块注册为子模块。正确的做法是使用 nn.ModuleList：
#普通列表中的模块通常不会被 PyTorch 正确注册。应该使用：
#self.layers = nn.ModuleList([
#    nn.Linear(20, 256),
#    nn.ReLU(),
#    nn.Linear(256, 10)
#])

#调用示例
#class MyNetwork(nn.Module):
#    def __init__(self):
#        super().__init__()
#        self.layers = nn.ModuleList([
#            nn.Linear(20, 256),
#            nn.ReLU(),
#            nn.Linear(256, 10)
#        ])

#    def forward(self, X):
#        for layer in self.layers:
#            X = layer(X)
#        return X

`__init__`函数将每个模块逐个添加到有序字典`_modules`中。
读者可能会好奇为什么每个`Module`都有一个`_modules`属性？
以及为什么我们使用它而不是自己定义一个Python列表？
简而言之，`_modules`的主要优点是：
在模块的参数初始化过程中，
系统知道在`_modules`字典中查找需要初始化参数的子块。


当`MySequential`的前向传播函数被调用时，
每个添加的块都按照它们被添加的顺序执行。
现在可以使用我们的`MySequential`类重新实现多层感知机。


In [47]:
net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)

tensor([[-5.7005e-02,  1.1483e-04,  8.1779e-02, -2.1998e-01,  3.6401e-01,
         -2.1379e-01,  1.0062e-01,  1.1973e-01,  8.0783e-02,  1.7940e-01],
        [-4.1021e-04,  3.8634e-02,  4.2088e-02, -2.6052e-01,  3.9457e-01,
         -1.6722e-01,  1.7996e-01,  2.2825e-01,  3.8820e-02,  1.4720e-01]],
       grad_fn=<AddmmBackward0>)

请注意，`MySequential`的用法与之前为`Sequential`类编写的代码相同
（如 :numref:`sec_mlp_concise` 中所述）。

## [**在前向传播函数中执行代码**]

`Sequential`类使模型构造变得简单，
允许我们组合新的架构，而不必定义自己的类。
然而，并不是所有的架构都是简单的顺序架构。
当需要更强的灵活性时，我们需要定义自己的块。
例如，我们可能希望在前向传播函数中执行Python的控制流。
此外，我们可能希望执行任意的数学运算，
而不是简单地依赖预定义的神经网络层。

到目前为止，
我们网络中的所有操作都对网络的激活值及网络的参数起作用。
然而，有时我们可能希望合并既不是上一层的结果也不是可更新参数的项，
我们称之为*常数参数*（constant parameter）。
例如，我们需要一个计算函数
$f(\mathbf{x},\mathbf{w}) = c \cdot \mathbf{w}^\top \mathbf{x}$的层，
其中$\mathbf{x}$是输入，
$\mathbf{w}$是参数，
$c$是某个在优化过程中没有更新的指定常量。
因此我们实现了一个`FixedHiddenMLP`类，如下所示：


In [51]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # 不计算梯度的随机权重参数。因此其在训练期间保持不变
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)
        # 使用创建的常量参数以及relu和mm函数
        X = F.relu(torch.mm(X, self.rand_weight) + 1)  ##广播机制
        # 复用全连接层。这相当于两个全连接层共享参数
        X = self.linear(X)
        # 控制流
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

In [52]:
#self.rand_weight = torch.rand((20, 20), requires_grad=False)
# 表示这个张量不需要计算梯度，因此训练时不会更新。
#它只在创建模型时随机生成一次。之后每次前向传播都使用同一个固定矩阵。
#不过，这里的 rand_weight 严格来说不是“模型参数”，只是保存在模型对象中的普通张量：
dict(net.named_parameters())

{'0.weight': Parameter containing:
 tensor([[-0.1664, -0.1695, -0.0223,  ...,  0.1459, -0.0798,  0.0319],
         [-0.1276, -0.0616, -0.1013,  ..., -0.1159, -0.1085,  0.1018],
         [-0.0473, -0.0353,  0.0257,  ..., -0.1254, -0.1240, -0.0740],
         ...,
         [ 0.0968,  0.1316,  0.1283,  ..., -0.1104,  0.0699,  0.0771],
         [ 0.1000, -0.1377,  0.1217,  ...,  0.0456,  0.0578,  0.1994],
         [ 0.1626,  0.0167,  0.1318,  ...,  0.1198, -0.0465,  0.0433]],
        requires_grad=True),
 '0.bias': Parameter containing:
 tensor([ 0.1654, -0.1719,  0.2104, -0.1986,  0.0412,  0.2196, -0.1386,  0.1870,
         -0.1729,  0.1885, -0.0374,  0.1034, -0.1631, -0.2168,  0.0128,  0.1883,
          0.0946,  0.1801, -0.0628,  0.1974, -0.1984,  0.1905, -0.1806,  0.0779,
         -0.1976,  0.0586, -0.0257, -0.1626,  0.1483, -0.1450,  0.1597,  0.1244,
          0.1250,  0.1723, -0.0658, -0.1183, -0.1567, -0.1646, -0.0208,  0.1769,
         -0.0791, -0.0956, -0.0101,  0.0124,  0.0686,  0.

在这个`FixedHiddenMLP`模型中，我们实现了一个隐藏层，
其权重（`self.rand_weight`）在实例化时被随机初始化，之后为常量。
这个权重不是一个模型参数，因此它永远不会被反向传播更新。
然后，神经网络将这个固定层的输出通过一个全连接层。

注意，在返回输出之前，模型做了一些不寻常的事情：
它运行了一个while循环，在$L_1$范数大于$1$的条件下，
将输出向量除以$2$，直到它满足条件为止。
最后，模型返回了`X`中所有项的和。
注意，此操作可能不会常用于在任何实际任务中，
我们只展示如何将任意代码集成到神经网络计算的流程中。


In [53]:
net = FixedHiddenMLP()
net(X)

tensor(-0.4349, grad_fn=<SumBackward0>)

我们可以[**混合搭配各种组合块的方法**]。
在下面的例子中，我们以一些想到的方法嵌套块。


In [55]:
class NestMLP(nn.Module):                                    ##内层嵌套块：嵌套了两个线性层和一个ReLU层
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(),
                                 nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)

    def forward(self, X):
        return self.linear(self.net(X))

chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())   ##外层嵌套块：嵌套了两个NestMLP块和一个FixedHiddenMLP块
chimera(X)

tensor(-0.2415, grad_fn=<SumBackward0>)

In [58]:
print (chimera)

Sequential(
  (0): NestMLP(
    (net): Sequential(
      (0): Linear(in_features=20, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=32, bias=True)
      (3): ReLU()
    )
    (linear): Linear(in_features=32, out_features=16, bias=True)
  )
  (1): Linear(in_features=16, out_features=20, bias=True)
  (2): FixedHiddenMLP(
    (linear): Linear(in_features=20, out_features=20, bias=True)
  )
)


In [60]:
print(chimera[1].weight.shape)
# torch.Size([20, 16])

print(chimera[2].linear.weight.shape)
# torch.Size([20, 20])

torch.Size([20, 16])
torch.Size([20, 20])


## 效率


读者可能会开始担心操作效率的问题。
毕竟，我们在一个高性能的深度学习库中进行了大量的字典查找、
代码执行和许多其他的Python代码。
Python的问题[全局解释器锁](https://wiki.python.org/moin/GlobalInterpreterLock)
是众所周知的。
在深度学习环境中，我们担心速度极快的GPU可能要等到CPU运行Python代码后才能运行另一个作业。


## 小结

* 一个块可以由许多层组成；一个块可以由许多块组成。
* 块可以包含代码。
* 块负责大量的内部处理，包括参数初始化和反向传播。
* 层和块的顺序连接由`Sequential`块处理。

## 练习

1. 如果将`MySequential`中存储块的方式更改为Python列表，会出现什么样的问题？
1. 实现一个块，它以两个块为参数，例如`net1`和`net2`，并返回前向传播中两个网络的串联输出。这也被称为平行块。
1. 假设我们想要连接同一网络的多个实例。实现一个函数，该函数生成同一个块的多个实例，并在此基础上构建更大的网络。


[Discussions](https://discuss.d2l.ai/t/1827)
